### sobolev kernel evalustion

In [1]:
import time
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
from sklearn import metrics
from sklearn.metrics.pairwise import rbf_kernel

from goodpoints import compress
from openxai.model import LoadModel
from openxai.dataloader import ReturnLoaders
import sage
import shap

In [2]:
_, loader_test = ReturnLoaders(data_name="german", download=False, batch_size=128)
X_test = loader_test.dataset.data
y_test = loader_test.dataset.targets.to_numpy()
model = LoadModel(data_name="german", ml_model="ann", pretrained=True)
model.eval()
predictions = model.predict(X_test)

### compression

In [3]:
start_time = time.time()
compressed_indices_sobolev1 = compress.compresspp_kt(X_test, kernel_type=b"sobolev", k_params=np.array([1.0]), seed=1)
compressed_indices_sobolev1_time = time.time() - start_time

In [ ]:
X_test_sobolev1 = X_test[compressed_indices_sobolev1]
y_test_sobolev1 = y_test[compressed_indices_sobolev1]

In [20]:
start_time = time.time()
compressed_indices_sobolev3 = compress.compresspp_kt(X_test, kernel_type=b"sobolev", k_params=np.array([3.0]), seed=1)
compressed_indices_sobolev3_time = time.time() - start_time
X_test_sobolev3 = X_test[compressed_indices_sobolev3]
y_test_sobolev3 = y_test[compressed_indices_sobolev3]

### explain

In [21]:
start_time = time.time()
explainer = shap.KernelExplainer(lambda x: model.predict_proba(x)[:, 1], X_test_sobolev1, seed=1)
exp_shap_sobolev1 = explainer(X_test, silent=True).values
end_time_explanation_shap_sobolev1 = time.time() - start_time

start_time = time.time()
imputer = sage.MarginalImputer(model.predict_proba, X_test_sobolev1)
explainer = (sage.PermutationEstimator)(imputer, loss="cross entropy", random_state=1)
exp_sage_sobolev1 = explainer(X_test, y_test, bar=False, verbose=True).values
end_time_explanation_sage_sobolev1 = time.time() - start_time

StdDev Ratio = 0.1489 (Converge at 0.0250)
StdDev Ratio = 0.1073 (Converge at 0.0250)
StdDev Ratio = 0.0834 (Converge at 0.0250)
StdDev Ratio = 0.0781 (Converge at 0.0250)
StdDev Ratio = 0.0661 (Converge at 0.0250)
StdDev Ratio = 0.0615 (Converge at 0.0250)
StdDev Ratio = 0.0575 (Converge at 0.0250)
StdDev Ratio = 0.0536 (Converge at 0.0250)
StdDev Ratio = 0.0495 (Converge at 0.0250)
StdDev Ratio = 0.0470 (Converge at 0.0250)
StdDev Ratio = 0.0459 (Converge at 0.0250)
StdDev Ratio = 0.0435 (Converge at 0.0250)
StdDev Ratio = 0.0423 (Converge at 0.0250)
StdDev Ratio = 0.0410 (Converge at 0.0250)
StdDev Ratio = 0.0397 (Converge at 0.0250)
StdDev Ratio = 0.0380 (Converge at 0.0250)
StdDev Ratio = 0.0362 (Converge at 0.0250)
StdDev Ratio = 0.0349 (Converge at 0.0250)
StdDev Ratio = 0.0344 (Converge at 0.0250)
StdDev Ratio = 0.0333 (Converge at 0.0250)
StdDev Ratio = 0.0322 (Converge at 0.0250)
StdDev Ratio = 0.0319 (Converge at 0.0250)
StdDev Ratio = 0.0313 (Converge at 0.0250)
StdDev Rati

In [ ]:
start_time = time.time()
explainer = shap.KernelExplainer(lambda x: model.predict_proba(x)[:, 1], X_test_sobolev3, seed=1)
exp_shap_sobolev3 = explainer(X_test, silent=True).values
end_time_explanation_shap_sobolev3 = time.time() - start_time

start_time = time.time()
imputer = sage.MarginalImputer(model.predict_proba, X_test_sobolev3)
explainer = (sage.PermutationEstimator)(imputer, loss="cross entropy", random_state=1)
exp_sage_sobolev3 = explainer(X_test, y_test, bar=False, verbose=True).values
end_time_explanation_sage_sobolev3 = time.time() - start_time

### calculate metric

In [17]:
def calculate_mmd(X, Y, gamma):
        XX = rbf_kernel(X, X, gamma)
        YY = rbf_kernel(Y, Y, gamma)
        XY = rbf_kernel(X, Y, gamma)
        return XX.mean() + YY.mean() - 2 * XY.mean()
gamma_val = 1 / (np.sqrt(2 * X_test.shape[1])**2)
def calculate_top_k(exp, gt, k=5):
        if exp.ndim == 2:  
                exp = np.mean(exp, axis=0)
        if gt.ndim == 2:
                gt = np.mean(gt, axis=0)

        exp_top_k = np.argsort(exp)[-k:]  
        gt_top_k = np.argsort(gt)[-k:]  
        correct_top_k = len(set(exp_top_k).intersection(set(gt_top_k)))
        return correct_top_k / k

In [18]:
values_gt = np.load(f'../metadata/german/ground_truth_shap_sage_results_0_1.npy', allow_pickle=True).item()
mmd_sobolev1 = calculate_mmd(X_test, X_test_sobolev1, gamma=gamma_val)
mae = None
top_k_score = None
gt_key_shap = 'ground_truth_shap_kernel'
gt_key_sage = 'ground_truth_sage_permutation'
if gt_key_shap in values_gt:
    values_gt_shap_sobolev1 = np.mean(np.array([values_gt[gt_key_shap][i] for i in range(1)]), axis=0)
    mae_shap_sobolev1 = np.mean(np.abs(exp_shap_sobolev1 - values_gt_shap_sobolev1))
    top_k_score_shap_sobolev1 = calculate_top_k(exp_shap_sobolev1, values_gt_shap_sobolev1, k=5)
if gt_key_sage in values_gt:
    values_gt_sage_sobolev1 = np.mean(np.array([values_gt[gt_key_sage][i] for i in range(1)]), axis=0)
    mae_sage_sobolev1 = np.mean(np.abs(exp_sage_sobolev1 - values_gt_sage_sobolev1))
    top_k_score_sage_sobolev1 = calculate_top_k(exp_sage_sobolev1, values_gt_sage_sobolev1, k=5)

row1 = {
    'dataset': "german",
    'repeat': 1,
    'method': f'cte_sobolev_1_shap', 
    'time_explanation': end_time_explanation_shap_sobolev1,
    'time_sample_choose': compressed_indices_sobolev1_time, 
    'n_original': X_test.shape[0],
    'n_sample': len(compressed_indices_sobolev1),
    'mmd': mmd_sobolev1,
    'mae': mae_shap_sobolev1,
    'top_k': top_k_score_shap_sobolev1
}
row2 = {
    'dataset': "german",
    'repeat': 1,
    'method': f'cte_sobolev_1_sage', 
    'time_explanation': end_time_explanation_sage_sobolev1,
    'time_sample_choose': compressed_indices_sobolev1_time, 
    'n_original': X_test.shape[0],
    'n_sample': len(compressed_indices_sobolev1),
    'mmd': mmd_sobolev1,
    'mae': mae_sage_sobolev1,
    'top_k': top_k_score_sage_sobolev1
}
times_sobolev1 = pd.concat([pd.DataFrame([row1]), pd.DataFrame([row2])])


In [23]:
mmd_sobolev3 = calculate_mmd(X_test, X_test_sobolev3, gamma=gamma_val)
mae = None
top_k_score = None
gt_key_shap = 'ground_truth_shap_kernel'
gt_key_sage = 'ground_truth_sage_permutation'
if gt_key_shap in values_gt:
    values_gt_shap_sobolev3 = np.mean(np.array([values_gt[gt_key_shap][i] for i in range(1)]), axis=0)
    mae_shap_sobolev3 = np.mean(np.abs(exp_shap_sobolev3 - values_gt_shap_sobolev3))
    top_k_score_shap_sobolev3 = calculate_top_k(exp_shap_sobolev3, values_gt_shap_sobolev3, k=5)
if gt_key_sage in values_gt:
    values_gt_sage_sobolev3 = np.mean(np.array([values_gt[gt_key_sage][i] for i in range(1)]), axis=0)
    mae_sage_sobolev3 = np.mean(np.abs(exp_sage_sobolev3 - values_gt_sage_sobolev3))
    top_k_score_sage_sobolev3 = calculate_top_k(exp_sage_sobolev3, values_gt_sage_sobolev3, k=5)

row1 = {
    'dataset': "german",
    'repeat': 1,
    'method': f'cte_sobolev_3_shap', 
    'time_explanation': end_time_explanation_shap_sobolev3,
    'time_sample_choose': compressed_indices_sobolev3_time, 
    'n_original': X_test.shape[0],
    'n_sample': len(compressed_indices_sobolev3),
    'mmd': mmd_sobolev3,
    'mae': mae_shap_sobolev3,
    'top_k': top_k_score_shap_sobolev3
}
row2 = {
    'dataset': "german",
    'repeat': 1,
    'method': f'cte_sobolev_3_sage', 
    'time_explanation': end_time_explanation_sage_sobolev3,
    'time_sample_choose': compressed_indices_sobolev3_time, 
    'n_original': X_test.shape[0],
    'n_sample': len(compressed_indices_sobolev3),
    'mmd': mmd_sobolev3,
    'mae': mae_sage_sobolev3,
    'top_k': top_k_score_sage_sobolev3
}
times_sobolev3 = pd.concat([pd.DataFrame([row1]), pd.DataFrame([row2])])


In [19]:
times_sobolev1

,dataset,repeat,method,time_explanation,time_sample_choose,n_original,n_sample,mmd,mae,top_k
0,german,1,cte_sobolev_1_shap,26.847223,0.003784,200,8,0.016532,0.011460,0.0
0,german,1,cte_sobolev_1_sage,94.199718,0.003784,200,8,0.016532,0.008553,0.4


In [24]:
times_sobolev3

,dataset,repeat,method,time_explanation,time_sample_choose,n_original,n_sample,mmd,mae,top_k
0,german,1,cte_sobolev_3_shap,26.354104,0.001106,200,8,0.028271,0.013507,0.2
0,german,1,cte_sobolev_3_sage,49.904897,0.001106,200,8,0.028271,0.015235,0.0


In [ ]:
dataset repeat         method       time_explanation   time_sample_choose  n_original  n_sample  mmd       mae        top_k
german   0           cte_shap           27.528821         0.00154614            200        8   0.0042569  0.0085143    0.0
german   0           cte_sage           71.990056         0.00154614            200        8   0.0042569  0.0119092    0.2

only gain - faster time_sample_choose, time_explanation for k=3, and lower mae for sage for k=1

### compas

In [25]:
_, loader_test = ReturnLoaders(data_name="compas", download=False, batch_size=128)
X_test = loader_test.dataset.data
y_test = loader_test.dataset.targets.to_numpy()
model = LoadModel(data_name="compas", ml_model="ann", pretrained=True)
model.eval()
predictions = model.predict(X_test)


start_time = time.time()
compressed_indices_sobolev1 = compress.compresspp_kt(X_test, kernel_type=b"sobolev", k_params=np.array([1.0]), seed=1)
compressed_indices_sobolev1_time = time.time() - start_time
X_test_sobolev1 = X_test[compressed_indices_sobolev1]
y_test_sobolev1 = y_test[compressed_indices_sobolev1]

start_time = time.time()
compressed_indices_sobolev3 = compress.compresspp_kt(X_test, kernel_type=b"sobolev", k_params=np.array([3.0]), seed=1)
compressed_indices_sobolev3_time = time.time() - start_time
X_test_sobolev3 = X_test[compressed_indices_sobolev3]
y_test_sobolev3 = y_test[compressed_indices_sobolev3]


start_time = time.time()
explainer = shap.KernelExplainer(lambda x: model.predict_proba(x)[:, 1], X_test_sobolev1, seed=1)
exp_shap_sobolev1 = explainer(X_test, silent=True).values
end_time_explanation_shap_sobolev1 = time.time() - start_time

start_time = time.time()
imputer = sage.MarginalImputer(model.predict_proba, X_test_sobolev1)
explainer = (sage.PermutationEstimator)(imputer, loss="cross entropy", random_state=1)
exp_sage_sobolev1 = explainer(X_test, y_test, bar=False, verbose=True).values
end_time_explanation_sage_sobolev1 = time.time() - start_time

start_time = time.time()
explainer = shap.KernelExplainer(lambda x: model.predict_proba(x)[:, 1], X_test_sobolev3, seed=1)
exp_shap_sobolev3 = explainer(X_test, silent=True).values
end_time_explanation_shap_sobolev3 = time.time() - start_time

start_time = time.time()
imputer = sage.MarginalImputer(model.predict_proba, X_test_sobolev3)
explainer = (sage.PermutationEstimator)(imputer, loss="cross entropy", random_state=1)
exp_sage_sobolev3 = explainer(X_test, y_test, bar=False, verbose=True).values
end_time_explanation_sage_sobolev3 = time.time() - start_time


values_gt = np.load(f'../metadata/compas/ground_truth_shap_sage_results_0_3.npy', allow_pickle=True).item()

mmd_sobolev1 = calculate_mmd(X_test, X_test_sobolev1, gamma=gamma_val)
mae = None
top_k_score = None
gt_key_shap = 'ground_truth_shap_kernel'
gt_key_sage = 'ground_truth_sage_permutation'
if gt_key_shap in values_gt:
    values_gt_shap_sobolev1 = np.mean(np.array([values_gt[gt_key_shap][i] for i in range(3)]), axis=0)
    mae_shap_sobolev1 = np.mean(np.abs(exp_shap_sobolev1 - values_gt_shap_sobolev1))
    top_k_score_shap_sobolev1 = calculate_top_k(exp_shap_sobolev1, values_gt_shap_sobolev1, k=5)
if gt_key_sage in values_gt:
    values_gt_sage_sobolev1 = np.mean(np.array([values_gt[gt_key_sage][i] for i in range(3)]), axis=0)
    mae_sage_sobolev1 = np.mean(np.abs(exp_sage_sobolev1 - values_gt_sage_sobolev1))
    top_k_score_sage_sobolev1 = calculate_top_k(exp_sage_sobolev1, values_gt_sage_sobolev1, k=5)

row1 = {
    'dataset': "compas",
    'repeat': 1,
    'method': f'cte_sobolev_1_shap', 
    'time_explanation': end_time_explanation_shap_sobolev1,
    'time_sample_choose': compressed_indices_sobolev1_time, 
    'n_original': X_test.shape[0],
    'n_sample': len(compressed_indices_sobolev1),
    'mmd': mmd_sobolev1,
    'mae': mae_shap_sobolev1,
    'top_k': top_k_score_shap_sobolev1
}
row2 = {
    'dataset': "compas",
    'repeat': 1,
    'method': f'cte_sobolev_1_sage', 
    'time_explanation': end_time_explanation_sage_sobolev1,
    'time_sample_choose': compressed_indices_sobolev1_time, 
    'n_original': X_test.shape[0],
    'n_sample': len(compressed_indices_sobolev1),
    'mmd': mmd_sobolev1,
    'mae': mae_sage_sobolev1,
    'top_k': top_k_score_sage_sobolev1
}
times_sobolev1_compas = pd.concat([pd.DataFrame([row1]), pd.DataFrame([row2])])

mmd_sobolev3 = calculate_mmd(X_test, X_test_sobolev3, gamma=gamma_val)
mae = None
top_k_score = None
gt_key_shap = 'ground_truth_shap_kernel'
gt_key_sage = 'ground_truth_sage_permutation'
if gt_key_shap in values_gt:
    values_gt_shap_sobolev3 = np.mean(np.array([values_gt[gt_key_shap][i] for i in range(3)]), axis=0)
    mae_shap_sobolev3 = np.mean(np.abs(exp_shap_sobolev3 - values_gt_shap_sobolev3))
    top_k_score_shap_sobolev3 = calculate_top_k(exp_shap_sobolev3, values_gt_shap_sobolev3, k=5)
if gt_key_sage in values_gt:
    values_gt_sage_sobolev3 = np.mean(np.array([values_gt[gt_key_sage][i] for i in range(3)]), axis=0)
    mae_sage_sobolev3 = np.mean(np.abs(exp_sage_sobolev3 - values_gt_sage_sobolev3))
    top_k_score_sage_sobolev3 = calculate_top_k(exp_sage_sobolev3, values_gt_sage_sobolev3, k=5)

row1 = {
    'dataset': "compas",
    'repeat': 1,
    'method': f'cte_sobolev_3_shap', 
    'time_explanation': end_time_explanation_shap_sobolev3,
    'time_sample_choose': compressed_indices_sobolev3_time, 
    'n_original': X_test.shape[0],
    'n_sample': len(compressed_indices_sobolev3),
    'mmd': mmd_sobolev3,
    'mae': mae_shap_sobolev3,
    'top_k': top_k_score_shap_sobolev3
}
row2 = {
    'dataset': "compas",
    'repeat': 1,
    'method': f'cte_sobolev_3_sage', 
    'time_explanation': end_time_explanation_sage_sobolev3,
    'time_sample_choose': compressed_indices_sobolev3_time, 
    'n_original': X_test.shape[0],
    'n_sample': len(compressed_indices_sobolev3),
    'mmd': mmd_sobolev3,
    'mae': mae_sage_sobolev3,
    'top_k': top_k_score_sage_sobolev3
}
times_sobolev3_compas = pd.concat([pd.DataFrame([row1]), pd.DataFrame([row2])])


StdDev Ratio = 0.3359 (Converge at 0.0250)
StdDev Ratio = 0.2865 (Converge at 0.0250)
StdDev Ratio = 0.2088 (Converge at 0.0250)
StdDev Ratio = 0.1803 (Converge at 0.0250)
StdDev Ratio = 0.1842 (Converge at 0.0250)
StdDev Ratio = 0.1832 (Converge at 0.0250)
StdDev Ratio = 0.1756 (Converge at 0.0250)
StdDev Ratio = 0.1574 (Converge at 0.0250)
StdDev Ratio = 0.1543 (Converge at 0.0250)
StdDev Ratio = 0.1316 (Converge at 0.0250)
StdDev Ratio = 0.1253 (Converge at 0.0250)
StdDev Ratio = 0.1156 (Converge at 0.0250)
StdDev Ratio = 0.1146 (Converge at 0.0250)
StdDev Ratio = 0.1139 (Converge at 0.0250)
StdDev Ratio = 0.1119 (Converge at 0.0250)
StdDev Ratio = 0.1078 (Converge at 0.0250)
StdDev Ratio = 0.1066 (Converge at 0.0250)
StdDev Ratio = 0.1042 (Converge at 0.0250)
StdDev Ratio = 0.1013 (Converge at 0.0250)
StdDev Ratio = 0.1022 (Converge at 0.0250)
StdDev Ratio = 0.1027 (Converge at 0.0250)
StdDev Ratio = 0.1006 (Converge at 0.0250)
StdDev Ratio = 0.0990 (Converge at 0.0250)
StdDev Rati

In [26]:
times_sobolev1_compas

,dataset,repeat,method,time_explanation,time_sample_choose,n_original,n_sample,mmd,mae,top_k
0,compas,1,cte_sobolev_1_shap,25.986105,0.001179,1235,32,0.00029,0.004541,0.6
0,compas,1,cte_sobolev_1_sage,155.155672,0.001179,1235,32,0.00029,0.014366,1.0


In [27]:
times_sobolev3_compas

,dataset,repeat,method,time_explanation,time_sample_choose,n_original,n_sample,mmd,mae,top_k
0,compas,1,cte_sobolev_3_shap,25.869481,0.000784,1235,32,0.000846,0.008349,0.6
0,compas,1,cte_sobolev_3_sage,105.129433,0.000784,1235,32,0.000846,0.030518,0.8


In [ ]:
dataset repeat  method     time_explanation  time_sample_choose   n_original  n_sample   mmd            mae       top_k
compas   0       cte_shap   26.06677484     0.02332592            1235         32       0.000165079   0.00313477   0.6
compas   0       cte_sage   209.8438327     0.02332592            1235         32       0.000165079   0.01233301   1.0

gain - shorter explanation time, time_sample_choose especially for sage; lower mmd gor k=3, lower mae for sage k=3

### Theory 
The Sobolev kernel focuses on a limited function class (with potentially faster computation and simpler interpretation). The Gaussian kernel covers a much richer function class, potentially giving better compression fidelity overall, but in regimes where that extra smoothness isn’t needed, it might not yield significantly better results than the Sobolev kernel.